Skull density ratio calculation code for "Zero echo time imaging-based skull density ratio for high-intensity focused ultrasound"

Kälvälä, R., Blanco Sequeiros, R., Frantzén, J., & Sainio, T. (2026). 
Zero echo time imaging-based skull density ratio for high-intensity focused ultrasound. 
International Journal of Hyperthermia, 43(1). 
https://doi.org/10.1080/02656736.2026.2706720

Copyright (C) 2026 Reetta Kälvälä
This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.
This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program. If not, see <https://www.gnu.org/licenses/>.

### Skull density ratio calculation

Input: Segmented calvaria volume containing the skull volume of interest in RAS coordinates.

Output: Skull density ratio (SDR) calculated with 


$SDR = \frac{\sum_{k=1}^{p} \min(I_{\mathrm{k}}) / \max(I_{\mathrm{k}})}{P}$,

where $P$=1093 (by default) indicates the number of ultrasound beams from the transducer, $\min(I_{\mathrm{k}})$ is the minimum and $\max(I_{\mathrm{k}})$ is the maximum skull intensity along each beam accordingly.


In [ ]:
# ------------------------------------------------
# Packages and printing options
# ------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import SimpleITK as sitk
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage import measure
from scipy.ndimage import map_coordinates

%matplotlib inline

In [ ]:
# ------------------------------------------------
# Data and margin
# ------------------------------------------------

# Margin from the outer egges of the skull, where the intensity minimum will not selected. 
# This is done to exclude the minimum to be selected from the partial volume effect of the bone-air boundary. 
#Please select accordinly to your resolution.
fallback_margin = 6

# ------------------------------------------------
#Download the segmented calvaria volume
# ------------------------------------------------

# patient nro
nro='001'
# modality
mod = 'ZTE'

# skull mask file path, update accordingly
skull_mask_filename = 'Path_to_registered_and_segmented_masks/{}_{}_mask.nrrd'.format(nro, mod)

image=sitk.ReadImage(skull_mask_filename, sitk.sitkFloat32)
volume = sitk.GetArrayFromImage(image) 
spacing = image.GetSpacing()

In [ ]:
# ------------------------------------------------
# Create a sphere 
# ------------------------------------------------

def sphere(R, horizontal_split, vertical_split):
    theta = np.linspace(0, 2 * np.pi, horizontal_split, endpoint=False)
    phi = np.linspace(0, np.pi / 2, vertical_split + 1)
    c = np.cos(phi)
    s = np.sin(phi)
    x = R * np.outer(s, np.cos(theta))
    y = R * np.outer(s, np.sin(theta))
    z = R * np.outer(c, np.ones(horizontal_split))
    return x, y, z

In [ ]:
# ------------------------------------------------
# Create lines from origo to a spherical surface
# ------------------------------------------------
def create_multiple_line_markups(x, y, z, output_file, decimals=5):
    n_rows, n_cols = x.shape
    origin = (0.0, 0.0, 0.0)
    point_id = 0
    seen = set()
    markups_list = []
    data = []

    for i in range(n_rows):
        for j in range(n_cols):
            end_point = (
                round(float(x[i][j]), decimals),
                round(float(y[i][j]), decimals),
                round(float(z[i][j]), decimals)
            )
            if end_point in seen or end_point == origin:
                continue
            
            # unique lines dataframe
            seen.add(end_point)
            data.append({
                "label": f"L{point_id}",
                "start_x": origin[0], "start_y": origin[1], "start_z": origin[2],
                "end_x": end_point[0], "end_y": end_point[1], "end_z": end_point[2]
            })
            #point_id += 1

            # json markups file
            seen.add(end_point)
            markups_list.append({
                "type": "Line",
                "coordinateSystem": "LPS",
                "controlPoints": [
                    { "label": f"L{point_id}_start", "position": list(origin) },
                    { "label": f"L{point_id}_end", "position": list(end_point) }
                ]
            })
            point_id += 1
    # create json file for the lines to visualize in 3D slicer (optional)
    slicer_json = {
        "@schema": "https://raw.githubusercontent.com/slicer/slicer/master/Modules/Loadable/Markups/Resources/Schema/markups-schema-v1.0.0.json#",
        "markups": markups_list
    }

    # uncomment for more info
    '''with open(output_file, 'w') as f:
        json.dump(slicer_json, f, indent=2)
    print(f"Saved {len(markups_list)} lines to '{output_file}'")'''
    
    lines_df = pd.DataFrame(data)
    return lines_df


# The selected split will determine the amount of beams used in the SDR calculation
x, y, z = sphere(R=120, horizontal_split=52, vertical_split=21)

#line information of 1093 lines will be stored in lines_df with columns ['label', 'start_x', 'start_y', 'start_z', 'end_x', 'end_y', 'end_z']
lines_df= create_multiple_line_markups(x, y, z, output_file='hemisphere_lines_multi.mrk.json')
print(lines_df.shape[0])
print(lines_df.head(5))

In [ ]:
# ------------------------------------------------
# Visualization of the division of outer ends
# ------------------------------------------------
fig = plt.figure(figsize=(10, 10))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.scatter(x, y, z, color="darkslategrey", s=3)
ax1.set_title(f"Hemisphere with n={lines_df.shape[0]} points", fontsize=16)

ax1.set_xticks(np.arange(-150, 150, 50))
ax1.set_xlabel('x', labelpad=10)
ax1.set_yticks(np.arange(-150, 150, 50))
ax1.set_ylabel('y', labelpad=10)
ax1.set_zticks(np.arange(0, 150, 50))
ax1.set_zlabel('z', labelpad=2)
ax1.set_box_aspect([1, 1, 0.5])
plt.show()

In [ ]:
# ------------------------------------------------
# Overlay with calvaria segment
# ------------------------------------------------

spacing_original = image.GetSpacing()
volume_trans = np.transpose(volume, (2, 1, 0)) 
image_trans= sitk.GetImageFromArray(volume_trans)

#coordinate updates, change based on your need
image_trans.SetOrigin((-100, -115, -0.5))

spacing_transposed = (spacing_original[0], spacing_original[1], spacing_original[2])
image_trans.SetSpacing(spacing_transposed)
volume_final = sitk.GetArrayFromImage(image_trans) 
mask = (volume_final > 0).astype(np.uint8)

direction = np.array(image_trans.GetDirection()).reshape(3, 3)
spacing = np.array(image_trans.GetSpacing())
origin = np.array(image_trans.GetOrigin())
affine = direction @ np.diag(spacing)
affine = np.vstack([np.hstack([affine, origin.reshape(-1, 1)]), [0, 0, 0, 1]])

# Create mesh 
verts, faces, normals, values = measure.marching_cubes(volume_final, level=0.5, step_size=2)
verts = verts * spacing
verts = verts + np.array(image_trans.GetOrigin())

# Create the mesh data
mesh_data = verts[faces]

fig = plt.figure(figsize=(6,6))
ax1 = fig.add_subplot(projection='3d')
ax1.set_title("a. Alignment from above", pad=1, fontsize=16)
mesh1 = Poly3DCollection(mesh_data, alpha=0.3, color='darkgrey', edgecolor='none')
ax1.add_collection3d(mesh1)

# Plot all lines
for _, row in lines_df.iterrows():
    x = [row['start_x'], row['end_x']]
    y = [row['start_y'], row['end_y']]
    z = [row['start_z'], row['end_z']]
    ax1.plot(x, y, z, color='darkslategrey', linewidth=0.3)

ax1.set_box_aspect([1, 1, 0.75])
ax1.set_xlabel('x', labelpad=7)
ax1.set_ylabel('y', labelpad=7)
ax1.set_zlabel('z', labelpad=7)
xticks = np.arange(-150, 151, 50)
yticks = np.arange(-150, 151, 50)

ax1.set_xticks(xticks)
ax1.set_yticks(yticks)
# Only label -150, 0, 150
ax1.set_xticklabels([str(t) if t in (-150, 0, 150) else '' for t in xticks])
ax1.set_yticklabels([str(t) if t in (-150, 0, 150) else '' for t in yticks])
ax1.set_zticks([])
ax1.view_init(elev=90, azim=0)
plt.show()

In [ ]:
# Reuse the same mesh data
fig = plt.figure(figsize=(6,6))
ax2 = fig.add_subplot(projection='3d')
ax2.set_title("b. Alignment from the front", fontsize=16)
mesh2 = Poly3DCollection(mesh_data, alpha=0.5, color='darkgrey', edgecolor='none')
ax2.add_collection3d(mesh2)
for _, row in lines_df.iterrows():
    x = [row['start_x'], row['end_x']]
    y = [row['start_y'], row['end_y']]
    z = [row['start_z'], row['end_z']]
    ax2.plot(x, y, z, color='darkslategrey', linewidth=0.3)
ax2.set_box_aspect([1, 1, 0.5])
ax2.set_xlabel('x', labelpad=15)
ax2.set_xticks(np.arange(-150, 150, 50))
ax2.set_zticks(np.arange(0, 175, 50))
ax2.set_ylabel('y', labelpad=10)
ax2.set_yticks([])
ax2.set_zlabel('z')
ax2.view_init(elev=0, azim=90)
plt.show()

In [ ]:
# ------------------------------------------------
# Sample intensity values along a line with 'num_samples' samples
# ------------------------------------------------

def sample_intensity_along_line(
    start,
    end,
    image_data,
    affine,
    num_samples
):
    # Create points along line in world (RAS) space
    points_world = np.linspace(start, end, num_samples)

    # Convert to voxel coordinates
    inv_affine = np.linalg.inv(affine)

    points_h = np.concatenate(
        [points_world, np.ones((num_samples, 1))],
        axis=1
    )

    # shape -> (3, N)
    points_voxel = (
        (inv_affine @ points_h.T).T[:, :3].T
    )

    # Interpolate image intensities
    intensities = map_coordinates(
        image_data,
        points_voxel,
        order=1,
        mode='nearest'
    )

    return intensities

In [ ]:
# ------------------------------------------------
# Start the intensity profile analysis along the samples line
# ------------------------------------------------

def analyze_intensity_profile(
    intensity_profile,
    fallback_margin
):

    intensity_profile = np.asarray(intensity_profile)
    global_max = np.max(intensity_profile)
    nonzero_indices = np.nonzero(
        intensity_profile
    )[0]

    if len(nonzero_indices) == 0:

        return {
            "local_max_idx": np.argmax(intensity_profile),
            "local_max_val": global_max,
            "local_min_idx": None,
            "local_min_val": global_max
        }

    first_nonzero = nonzero_indices[0]
    last_nonzero = nonzero_indices[-1]

    start = first_nonzero + fallback_margin
    end = last_nonzero - fallback_margin

    # EXACT original safety
    if end <= start:

        return {
            "local_max_idx": np.argmax(intensity_profile),
            "local_max_val": global_max,
            "local_min_idx": np.argmax(intensity_profile),
            "local_min_val": global_max
        }

    valid_indices = np.arange(start, end)

    if len(valid_indices) == 0:

        return {
            "local_max_idx": np.argmax(intensity_profile),
            "local_max_val": global_max,
            "local_min_idx": np.argmax(intensity_profile),
            "local_min_val": global_max
        }

    local_min_idx = valid_indices[
        np.argmin(
            intensity_profile[valid_indices]
        )
    ]

    local_min_val = intensity_profile[
        local_min_idx
    ]

    return {

        "local_max_idx":
            np.argmax(intensity_profile),

        "local_max_val":
            global_max,

        "local_min_idx":
            local_min_idx,

        "local_min_val":
            local_min_val
    }

In [ ]:
# ------------------------------------------------
# Analyze all line profiles
# ------------------------------------------------

def analyze_all_lines(
    image_trans,
    affine,
    lines_df,
    num_samples,
    fallback_margin
):

    data = sitk.GetArrayFromImage(image_trans)

    results = []

    # Separate cache for profiles
    profiles = {}

    for _, row in lines_df.iterrows():

        start = np.array([
            row['start_x'],
            row['start_y'],
            row['start_z']
        ])

        end = np.array([
            row['end_x'],
            row['end_y'],
            row['end_z']
        ])

        # ----------------------------------------
        # Sample profile
        # ----------------------------------------

        profile = sample_intensity_along_line(
            start,
            end,
            data,
            affine,
            num_samples
        )

        # Store separately
        profiles[row['label']] = profile

        # ----------------------------------------
        # Analyze profile
        # ----------------------------------------

        stats = analyze_intensity_profile(
            profile,
            fallback_margin
        )

        results.append({

            "label": row['label'],

            "global_max":
                stats["local_max_val"],

            "local_minima":
                stats["local_min_val"]

        })

    results_df = pd.DataFrame(results)

    return results_df, profiles

In [ ]:
# ------------------------------------------------
# Plot intensity profile through selected line "n"
# ------------------------------------------------

def plot_line_intensity_profile(
    line_index,
    image,
    affine,
    lines_df,
    num_samples,
    fallback_margin,
    profile_cache=None,
    figsize=(10,5)
):

    row = lines_df.iloc[line_index]

    # ------------------------------------------------
    # Use cached profile if available
    # ------------------------------------------------

    if profile_cache is not None:

        intensity_profile = profile_cache[
            row['label']
        ]

    else:
        # Otherwise resample

        data = sitk.GetArrayFromImage(image)

        start = np.array([
            row['start_x'],
            row['start_y'],
            row['start_z']
        ])

        end = np.array([
            row['end_x'],
            row['end_y'],
            row['end_z']
        ])

        intensity_profile = sample_intensity_along_line(
            start,
            end,
            data,
            affine,
            num_samples
        )

    # ------------------------------------------------
    # Analyze profile
    # ------------------------------------------------

    stats = analyze_intensity_profile(
    intensity_profile,
    fallback_margin
    )
    
    local_max_idx = stats["local_max_idx"]
    local_max_val = stats["local_max_val"]
    
    local_min_idx = stats["local_min_idx"]
    local_min_val = stats["local_min_val"]

    # ------------------------------------------------
    # PLOT
    # ------------------------------------------------

    plt.figure(figsize=figsize)

    plt.plot(
        intensity_profile,
        label='Intensity profile',
        color='xkcd:dark grey blue'
    )
    
    # Main maximum
    if local_max_idx is not None:
    
        plt.plot(
            local_max_idx,
            local_max_val,
            'C0o',
            label='maximum'
        )
    
    # Main minimum
    if local_min_idx is not None:
    
        plt.plot(
            local_min_idx,
            local_min_val,
            'C4o',
            label='minimum'
        )
    
    plt.title(
        f"Intensity Profile along line {row['label']}"
    )
    plt.xlabel("Sample Index Along Line")
    plt.ylabel("Intensity")
    plt.grid(True)
    plt.legend()
    plt.show()

results_df, profiles = analyze_all_lines(
    image_trans,
    affine,
    lines_df,
    num_samples=500,
    fallback_margin=fallback_margin
)

merged_df = pd.merge(
    lines_df,
    results_df,
    on='label'
)

# number of the line
n = 5

plot_line_intensity_profile(
    n,
    image_trans,
    affine,
    lines_df,
    num_samples=500,
    fallback_margin=fallback_margin,
    profile_cache=profiles
)

In [ ]:
# ------------------------------------------------
# SDR CALCULATION
# ------------------------------------------------


# Ensure there are no divisions by zero
valid_rows = merged_df[(merged_df['global_max'] > 0) & (merged_df['local_minima'] > 0)] #!=

# Compute the SDR value
sdr = (valid_rows['local_minima'] / valid_rows['global_max']).mean()

print(f"Patient nro: {nro}")
print(f"Modality: {mod}\n")

# If there is no problematic rows
if len(valid_rows) == len(lines_df):

    print(f"Skull density ratio: {sdr:.4f}\n")

# ------------------------------------------------
# Print invalid rows to check further
# ------------------------------------------------

else:

    print(
        f"Skull density ratio: "
        f"{sdr:.4f} with "
        f"{len(valid_rows)}/{lines_df.shape[0]} "
        f"valid rows\n"
    )

    invalid_rows = merged_df.loc[
        merged_df['local_minima'] <= 0,
        [
            'label',
            'global_max',
            'local_minima'
        ]
    ]

    print("Rows with zero or negative minima:\n")
    print(invalid_rows.to_markdown())